# Tata Steel — Roller-Table Motor Predictive & Prescriptive Maintenance

**Motor ID:** ROT-MILL-05 | **Process:** TSCR Roller Table | **Sampling Rate:** 1 second

### Pipeline:
1. Full exploratory analysis of all 10 sensor columns
2. Statistical threshold derivation and feature-selection justification
3. ML model training, cross-validation, and benchmarking
4. Health-index scoring and sensor-aware prescriptive maintenance

**Dataset:** `tata_steel_rot_motor_proxy.csv` — 10,000 rows, 1-second intervals, simulated IoT data for a 415 V / 30 kW roller-table motor.

## Part 1 — Exploratory Data Analysis

We load all columns first, understand the data structure, then use correlation and distribution analysis to decide which features matter for modelling.

### 1.1 Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("tata_steel_rot_motor_proxy.csv")

print(f"Shape : {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n--- Data Types & Nulls ---")
df.info()
print("\n--- Descriptive Statistics (all numeric) ---")
df.describe().round(2)

### 1.2 Correlation Heatmap — All Numeric Sensors

This tells us which sensors move together (shared cause) and which are independent noise. We use this to justify feature selection for the ML model.

In [ ]:
numeric_cols = ["Current_Amp", "Voltage_V", "Motor_RPM", "Vibration_mm_s",
                "Winding_Temp_C", "Bearing_Temp_C", "Coolant_Pressure_Bar", "Ambient_Humidity_Pct"]

plt.figure(figsize=(9, 7))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5, square=True
)
plt.title("Sensor Correlation Matrix (all 8 numeric columns)", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

**Observations:**
- **Current, RPM, Vibration, and both Temperatures** are correlated — they all respond to the cyclic slab-loading pattern.
- **Voltage, Coolant Pressure, and Humidity** show near-zero correlation with everything else — independent environmental noise.

**Feature selection decision:** `Coolant_Pressure_Bar` and `Ambient_Humidity_Pct` are excluded from the ML feature set. `Voltage` is kept because supply fluctuations can directly affect current draw.

### 1.3 Data Visualization

The motor cycles between idling (45 A) and loaded (85 A) as steel slabs pass.

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

axes.plot(df["Current_Amp"].iloc[:500], color="#5B9BD5", linewidth=0.8)
axes.axhline(60, color="gray", linestyle=":", alpha=0.6, label="~60 A midpoint")
axes.set_xlabel("Time Index (seconds)")
axes.set_ylabel("Current (A)")
axes.set_title("Current Over Time — Cyclic Load Switching (first 500 s)")
axes.legend()

plt.tight_layout()
plt.show()

### 1.4 Temperature as a Lagging Indicator

Temperatures change slowly — they smooth towards a target rather than jumping instantly with load. Winding temp responds faster than bearing temp. This makes them good for tracking steady-state health, but not for detecting sudden faults — that role belongs to vibration.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = 800

axes[0].plot(df["Current_Amp"].iloc[:t],    color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[0].plot(df["Winding_Temp_C"].iloc[:t], color="#ED7D31", linewidth=1.2, label="Winding Temp (C)")
axes[0].set_xlabel("Time Index (seconds)")
axes[0].set_ylabel("Value")
axes[0].set_title("Current vs Winding Temperature — Lagging Response")
axes[0].legend(loc="upper right")

axes[1].plot(df["Current_Amp"].iloc[:t],     color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[1].plot(df["Bearing_Temp_C"].iloc[:t],  color="#70AD47", linewidth=1.2, label="Bearing Temp (C)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Value")
axes[1].set_title("Current vs Bearing Temperature — Even Slower Response")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 1.5 Rule-Based Anomaly Detection — Vibration Spikes

We can detect simple anomalies by checking if vibration levels exceed a fixed threshold (e.g., 7.0 mm/s). This provides a quick baseline for identifying potential mechanical issues.

- **Bearing Temp upper bound:** 85.50 °C
- **Winding Temp upper bound:** 105.61 °C

In [ ]:
# ---- Simple Anomaly Detection (Rule-Based) ----
vibration_threshold = 7.0  # mm/s (abnormally high)

anomalies = df[df["Vibration_mm_s"] > vibration_threshold]

print("Number of vibration anomalies detected:", len(anomalies))
print(anomalies[["Timestamp", "Vibration_mm_s"]].head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

# Full timeline
axes[0].plot(df["Vibration_mm_s"], color="#5B9BD5", linewidth=0.6, label="Vibration (mm/s)")
axes[0].scatter(anomalies.index, anomalies["Vibration_mm_s"],
                color="red", s=20, zorder=5, label="Anomaly (spike)")
axes[0].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"Threshold ({vibration_threshold:.2f})")
axes[0].set_title("Vibration Monitoring — Full Timeline")
axes[0].set_ylabel("Vibration (mm/s)")
axes[0].legend()

# Zoomed
z = 500
zoom_anom = anomalies[anomalies.index < z]
axes[1].plot(df["Vibration_mm_s"].iloc[:z], color="#5B9BD5", linewidth=0.9, label="Vibration (mm/s)")
axes[1].scatter(zoom_anom.index, zoom_anom["Vibration_mm_s"],
                color="red", s=45, zorder=5, label="Anomaly (spike)")
axes[1].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"Threshold ({vibration_threshold:.2f})")
axes[1].set_title(f"Vibration Monitoring — Zoomed (first {z} s)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Vibration (mm/s)")
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Part 2 — Machine Learning Model

We will train models to predict what the "normal" current should be. If the actual current deviates significantly from the prediction, it may indicate an issue with the motor.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Feature Selection
features = ["Motor_RPM", "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C", "Voltage_V"]
X = df[features]
y = df["Current_Amp"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Training set size:", len(X_train))
print("Test set size:", len(X_test))

### 2.1 Model Training

We compare **Linear Regression** (baseline), **Decision Tree**, and **Random Forest** using multiple regression metrics: MAE, RMSE, and R².

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = {}

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    results[name] = [mae, rmse, r2]

    print(f"\n{name}")
    print("MAE  :", round(mae, 3))
    print("RMSE :", round(rmse, 3))
    print("R2   :", round(r2, 3))

# Store Random Forest predictions for later use
rf_model = models["Random Forest"]
y_pred_rf = rf_model.predict(X_test)

### 2.2 Model Comparison Table

A consolidated view of all model performances for easy comparison.

In [ ]:
import pandas as pd

metrics_df = pd.DataFrame(results, index=["MAE", "RMSE", "R2"]).T
metrics_df

### Model Selection

We evaluated **Linear Regression**, **Decision Tree** and **Random Forest** using MAE, RMSE, and R².

**Random Forest** achieved the lowest error and highest R², indicating better modeling of nonlinear industrial sensor relationships between motor current and sensor features.

**Conclusion:** Random Forest is selected as the primary model for anomaly detection and health scoring.

---

## Part 3 — Feature Engineering

We create domain-specific engineered features to capture electrical, thermal, mechanical, and environmental behavior patterns that may indicate anomalies or degradation.

In [ ]:
import numpy as np
import pandas as pd

fe_df = df.copy()

# ---------------- ELECTRICAL ----------------
fe_df["Power_Index"] = fe_df["Voltage_V"] * fe_df["Current_Amp"]
fe_df["Current_Deviation"] = fe_df["Current_Amp"] - fe_df["Current_Amp"].rolling(window=30).mean()

# ---------------- THERMAL ----------------
fe_df["Thermal_Gradient"] = fe_df["Winding_Temp_C"] - fe_df["Bearing_Temp_C"]
fe_df["Winding_Temp_Rate"] = fe_df["Winding_Temp_C"].diff()

# ---------------- MECHANICAL ----------------
fe_df["Vibration_RPM_Ratio"] = fe_df["Vibration_mm_s"] / fe_df["Motor_RPM"]
fe_df["Vibration_Energy"] = fe_df["Vibration_mm_s"].rolling(window=20).mean()

# ---------------- COOLING / ENV ----------------
fe_df["Cooling_Efficiency"] = fe_df["Coolant_Pressure_Bar"] / fe_df["Winding_Temp_C"]
fe_df["Humidity_Thermal_Stress"] = fe_df["Ambient_Humidity_Pct"] * fe_df["Winding_Temp_C"]

fe_df = fe_df.dropna().reset_index(drop=True)

### 3.1 Learning Feature Importance

We train a Random Forest model to predict anomaly risk and extract feature importances to understand which engineered features contribute most to anomaly detection.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Target = anomaly risk proxy
y_anomaly = abs(fe_df["Current_Amp"] - fe_df["Current_Amp"].rolling(5).mean())
y_anomaly = y_anomaly.dropna().reset_index(drop=True)

X_anomaly = fe_df.iloc[y_anomaly.index].drop(columns=["Timestamp", "Motor_ID"])

# Train-test split
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_anomaly, y_anomaly, test_size=0.2, random_state=42
)

rf_anomaly = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_anomaly.fit(X_train_fe, y_train_fe)

### 3.2 Plotting Feature Importance

Visualize which features have the highest predictive power for identifying anomalies.

In [ ]:
importances = rf_anomaly.feature_importances_
feature_names = X_anomaly.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.gca().invert_yaxis()
plt.title("Feature Importance for Anomaly Risk Evaluation")
plt.xlabel("Relative Importance")
plt.tight_layout()
plt.show()

### 3.3 Prescriptive Analysis

Based on engineered feature thresholds, we define rule-based prescriptive actions that recommend specific maintenance interventions.

In [ ]:
# Calculate quantile threshold once before applying function
power_index_95 = fe_df["Power_Index"].quantile(0.95)

def prescriptive_engineering_rules(row):
    actions = []

    if row["Vibration_Energy"] > 4:
        actions.append("Inspect bearings and lubrication")

    if row["Vibration_RPM_Ratio"] > 0.004:
        actions.append("Check shaft alignment")

    if row["Thermal_Gradient"] > 20:
        actions.append("Inspect insulation and heat dissipation")

    if row["Winding_Temp_Rate"] > 0.5:
        actions.append("Reduce load or enhance cooling")

    if row["Cooling_Efficiency"] < 0.03:
        actions.append("Inspect coolant flow and pressure")

    if row["Humidity_Thermal_Stress"] > 6000:
        actions.append("Improve ventilation and moisture control")

    if row["Current_Deviation"] > 10:
        actions.append("Inspect mechanical load fluctuation")

    if row["Power_Index"] > power_index_95:
        actions.append("Verify supply voltage and motor loading")

    return ", ".join(actions) if actions else "Normal operation"

fe_df["Prescriptive_Action"] = fe_df.apply(prescriptive_engineering_rules, axis=1)

print(fe_df["Prescriptive_Action"].value_counts().head(10))

---

## Part 4 — Evaluating Risk Scores

We compute a multivariate anomaly severity score using standardized features and train a Random Forest to predict risk levels.

### 4.1 Defining Feature Metrics

We standardize all relevant sensor and engineered features, then compute a Euclidean-distance-based anomaly severity score.

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Select all relevant features
feature_cols = [
    "Voltage_V", "Current_Amp", "Motor_RPM",
    "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C",
    "Coolant_Pressure_Bar", "Ambient_Humidity_Pct",
    "Power_Index", "Current_Deviation", "Thermal_Gradient",
    "Winding_Temp_Rate", "Vibration_RPM_Ratio",
    "Vibration_Energy", "Cooling_Efficiency",
    "Humidity_Thermal_Stress"
]

X_risk = fe_df[feature_cols]

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_risk)

# Multivariate anomaly severity (Euclidean distance)
anomaly_severity = np.linalg.norm(X_scaled, axis=1)

fe_df["Anomaly_Severity"] = anomaly_severity

### 4.2 Training Random Forest Using All Features

Train a deeper Random Forest model to predict anomaly severity from all engineered features.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_risk = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

rf_risk.fit(X_risk, fe_df["Anomaly_Severity"])

### 4.3 Predicting Anomaly Risk Scores

Generate predictions and normalize them to a 0–1 scale for interpretability.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

raw_risk = rf_risk.predict(X_risk)

# Normalize to 0–1
risk_scaler = MinMaxScaler()

fe_df["Anomaly_Risk_Score"] = risk_scaler.fit_transform(
    raw_risk.reshape(-1, 1)
)

### 4.4 Classifying Risk Levels

Categorize the continuous risk score into discrete levels: **Low**, **Medium**, and **High**.

In [ ]:
def risk_level(score):
    if score < 0.3:
        return "Low"
    elif score < 0.6:
        return "Medium"
    else:
        return "High"

fe_df["Anomaly_Risk_Level"] = fe_df["Anomaly_Risk_Score"].apply(risk_level)

### 4.5 Final Results

Display the consolidated output with key sensor readings, engineered features, and risk assessments.

In [ ]:
final_cols = [
    "Timestamp",
    "Motor_ID",

    # Raw sensor values
    "Current_Amp",
    "Motor_RPM",
    "Vibration_mm_s",
    "Winding_Temp_C",
    "Bearing_Temp_C",

    # Engineered features (important ones)
    "Thermal_Gradient",
    "Vibration_Energy",
    "Cooling_Efficiency",

    # Risk outputs
    "Anomaly_Risk_Score",
    "Anomaly_Risk_Level"
]

display(fe_df[final_cols].head(10))

### 4.6 Risk Level Distribution

Visualize the distribution of anomaly risk levels across the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Count category distribution
risk_counts = fe_df["Anomaly_Risk_Level"].value_counts()

# Bar plot
plt.figure(figsize=(6, 4))
plt.bar(risk_counts.index, risk_counts.values, color=["green", "orange", "red"])
plt.title("Anomaly Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Instances")
plt.tight_layout()
plt.show()

# Pie chart
plt.figure(figsize=(6, 6))
plt.pie(
    risk_counts.values,
    labels=risk_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["green", "orange", "red"]
)
plt.title("Anomaly Risk Level Percentage Distribution")
plt.tight_layout()
plt.show()